# Лабораторна робота 10.2
## Тема: Перенесення навчання з аугментацією та тонким налаштуванням (Horse-Human)

**Завдання 2:** 
1. Розпізнавання тестових зображень (5-10 шт).
2. Аугментація даних (Data Augmentation).
3. Тонке налаштування (Fine Tuning).

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import Model
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
from google.colab import files
from keras.preprocessing import image
import matplotlib.pyplot as plt

# Завантаження ваг
!wget --no-check-certificate \
    https://storage.googleapis.com/mledu-datasets/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5 \
    -O /tmp/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5

local_weights_file = '/tmp/inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5'

# Завантаження даних
!wget --no-check-certificate https://storage.googleapis.com/laurencemoroney-blog.appspot.com/horse-or-human.zip -O /tmp/horse-or-human.zip
!wget --no-check-certificate https://storage.googleapis.com/laurencemoroney-blog.appspot.com/validation-horse-or-human.zip -O /tmp/validation-horse-or-human.zip 

import zipfile
local_zip = '//tmp/horse-or-human.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp/training')
zip_ref.close()

local_zip = '//tmp/validation-horse-or-human.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/tmp/validation')
zip_ref.close()

train_dir = '/tmp/training'
validation_dir = '/tmp/validation'

## Частина 2: Аугментація даних

In [ ]:
# Додаємо параметри аугментації
train_datagen = ImageDataGenerator(
      rescale = 1./255.,
      rotation_range=40,
      width_shift_range=0.2,
      height_shift_range=0.2,
      shear_range=0.2,
      zoom_range=0.2,
      horizontal_flip=True,
      fill_mode='nearest')

# Валідація без аугментації
test_datagen = ImageDataGenerator( rescale = 1.0/255. )

# Змінюємо target_size на 300x300 як просили в завданні
train_generator = train_datagen.flow_from_directory(train_dir,
                                                    batch_size = 20,
                                                    class_mode = 'binary', 
                                                    target_size = (300, 300))

validation_generator =  test_datagen.flow_from_directory( validation_dir,
                                                          batch_size  = 20,
                                                          class_mode  = 'binary', 
                                                          target_size = (300, 300))

## Побудова моделі (Transfer Learning)

In [ ]:
pre_trained_model = InceptionV3(input_shape = (300, 300, 3), 
                                include_top = False, 
                                weights = None)

pre_trained_model.load_weights(local_weights_file)

for layer in pre_trained_model.layers:
  layer.trainable = False
  
last_layer = pre_trained_model.get_layer('mixed7')
print('last layer output shape: ', last_layer.output_shape)
last_output = last_layer.output

x = layers.Flatten()(last_output)
x = layers.Dense(1024, activation='relu')(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(1, activation='sigmoid')(x)

model = Model(pre_trained_model.input, x)
model.compile(optimizer = RMSprop(learning_rate=0.0001), 
            loss = 'binary_crossentropy', 
            metrics = ['accuracy'])

history = model.fit(
            train_generator,
            validation_data = validation_generator,
            steps_per_epoch = 50,
            epochs = 20,
            validation_steps = 50,
            verbose = 2)

## Частина 3: Fine Tuning (Тонке налаштування)

In [ ]:
# Розморожуємо більше шарів для Fine Tuning
from tensorflow.keras.optimizers import SGD

unfreeze = False

# Приклад розморожування останніх блоків Inception
# Подивимось структуру, щоб вирішити з якого шару розморожувати
# pre_trained_model.summary()

for layer in pre_trained_model.layers:
  if layer.name == 'mixed6':
    unfreeze = True
  if unfreeze:
    layer.trainable = True
  else:
    layer.trainable = False

# Потрібно перекомпілювати модель з дуже малим learning rate
model.compile(optimizer=SGD(learning_rate=0.00001, momentum=0.9),
              loss='binary_crossentropy',
              metrics=['accuracy'])

history_fine = model.fit(
      train_generator,
      steps_per_epoch = 50,
      epochs = 10,
      validation_data = validation_generator,
      validation_steps = 50,
      verbose=2)

## Тестування на власних зображеннях

In [ ]:
# Завантаження файлів для тестування
uploaded = files.upload()

for fn in uploaded.keys():
  # prediction
  path = '/content/' + fn
  img = image.load_img(path, target_size=(300, 300))
  x = image.img_to_array(img)
  x = np.expand_dims(x, axis=0)

  images = np.vstack([x])
  classes = model.predict(images, batch_size=10)
  print(classes[0])
  if classes[0]>0.5:
    print(fn + " is a human")
  else:
    print(fn + " is a horse")